# Lab 11: Can a Machine Explain Itself?

### Looking for the reasons behind an AI's decisions, with LIME, SHAP, and saliency maps

When a system decides something about a person, that they are likely to reoffend, that their writing belongs to one camp and not another, that a photo contains a cat, we usually want to know *why*. The "why" is where philosophy has spent a lot of time. There is the old distinction between a **reason** and a **cause**. There is the worry, going back at least to Hume and sharpened by modern psychology, that the reasons we give for our own behaviour are often stories we make up after the fact rather than the real springs of action.

This lab puts that worry to a machine. We will use three popular tools that each claim to tell you why a model made a prediction. Then we will ask the harder question, the one that actually belongs to you as philosophers: when the machine hands us a reason, should we believe it?

You will not need to write any code. Press play on each cell, move the sliders, try your own inputs, and answer the thinking questions as you go.

### How this notebook works

Three kinds of cell show up over and over:

- **Think first.** A short question that asks what you expect *before* you see the result. Make a guess. Being wrong on purpose is how the interesting cells earn their keep.
- **Play.** A cell with a slider, a dropdown, or a text box. Drag things around. Watch what changes and what stays the same.
- **What just happened.** A short plain-language explanation, followed by a philosophical angle worth chewing on.

Run the cells in order, top to bottom. If something looks stuck, run the cell again.

## Part 0: Setup

Press play on the next two cells and wait for "Ready". Nothing to read here.

In [ ]:
!pip install -q shap lime captum ipywidgets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import ipywidgets as widgets
from ipywidgets import interact, interact_manual

np.random.seed(0)
print("Ready. You can keep going.")

## Part 1: The reasons behind a verdict

There is a real tool called **COMPAS** that has been used in United States courtrooms to estimate how likely someone is to commit another crime within two years. Judges have seen its scores while deciding on bail and sentencing. In 2016 the newsroom ProPublica investigated it and argued that it was biased against Black defendants. You met this story in the fairness labs earlier in the course.

We are going to build a simple stand-in for that kind of tool, using the same public data, and then we will make it explain itself. The point is not the model. The point is the explanation, and whether it deserves our trust.

### Think first

Before you see anything, picture the facts a tool like this might use: someone's age, their number of prior convictions, the type of their current charge, their sex, their race.

- Which of these facts do you think *should* be allowed to influence a prediction about reoffending?
- Which would strike you as unfair to use, even if it happened to improve accuracy?

Hold your answer in mind. We will come back to it once the machine has shown its hand.

<details>
<summary><b>Hint:</b> which fact would you refuse even if dropping it made the tool less accurate?</summary>

Most people happily let prior convictions and age count, and feel uneasy about race and sex, even when the unfair fact would lift accuracy. That gap is the point. Accuracy and fairness pull in different directions here, and no amount of extra data closes it for you. Notice that "but it improves accuracy" is an argument about prediction, not about justice. Keep the facts you marked off-limits in mind. In a few cells SHAP will show you what the model actually leaned on, and your two lists may not match.

</details>

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw = pd.read_csv(URL)

# ProPublica's standard cleanup of the raw records
df = raw[(raw.days_b_screening_arrest <= 30) &
         (raw.days_b_screening_arrest >= -30) &
         (raw.is_recid != -1) &
         (raw.c_charge_degree != "O") &
         (raw.score_text != "N/A")].copy()

# Keep the three largest groups so the example stays readable
df = df[df["race"].isin(["African-American", "Caucasian", "Hispanic"])].copy()

# A short list of plain-language facts about each person
X = pd.DataFrame({
    "Age": df["age"].values,
    "Prior convictions": df["priors_count"].values,
    "Juvenile felonies": df["juv_fel_count"].values,
    "Juvenile misdemeanors": df["juv_misd_count"].values,
    "Current charge is a felony": (df["c_charge_degree"] == "F").astype(int).values,
    "Is male": (df["sex"] == "Male").astype(int).values,
    "Is African-American": (df["race"] == "African-American").astype(int).values,
    "Is Caucasian": (df["race"] == "Caucasian").astype(int).values,
    "Is Hispanic": (df["race"] == "Hispanic").astype(int).values,
})
y = df["two_year_recid"].values   # 1 means the person did reoffend within two years

idx = np.arange(len(X))
itr, ite = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)
Xtr, Xte, ytr, yte = X.iloc[itr], X.iloc[ite], y[itr], y[ite]
race_te = df["race"].values[ite]

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0)
model.fit(Xtr, ytr)

print(f"We have {len(X)} people in this dataset.")
print(f"The model agrees with the real two-year outcome about {model.score(Xte, yte):.0%} of the time.")
X.head()

### What just happened

We trained a model on roughly six thousand real cases. It looks at a handful of facts about a person and outputs a score between 0 and 1 for "likely to reoffend within two years". Notice that we deliberately *included* race as one of the facts it is allowed to see. That was a choice, and a loaded one. Keep it in the back of your mind.

Notice also what the accuracy number does *not* tell you. "Right about two thirds of the time" sounds fine until you ask: right for whom, and wrong in which direction?

### Think first

The model gives a score, but it does not say why. Suppose you wanted to know how much each fact pushed a given person's score up or down. How would you even measure that? What would a fair way of splitting the credit look like?

There is an elegant answer that comes straight out of cooperative game theory.

<details>
<summary><b>Hint:</b> what makes a split fair when the facts interact?</summary>

One natural idea is to change a fact, hold the rest fixed, and see how far the score moves. That is on the right track, but it hides a problem: facts interact, so how much one fact matters can depend on which others are already in play. A fair split has to reckon with every order the facts might be taken in, not just one. If you want the fairness to be principled rather than ad hoc, you need a rule that treats every fact even-handedly, and averaging a fact's contribution over all possible orders is exactly that rule. That is the trick the next cell borrows from game theory.

</details>

### SHAP, in one idea

Imagine a group of friends shares a taxi. They each get out at different stops, so the total fare depends on who is in the cab and how far each one goes. How do you split the bill fairly? The mathematician Lloyd Shapley worked out an answer in 1953: look at every possible order in which people could have joined the ride, and charge each person their average added cost across all those orders. That fair share is called a **Shapley value**.

**SHAP** applies the same accounting to a prediction. Each fact about a person is a "passenger". SHAP asks how much that fact adds to the score, averaged over every order in which the facts could be taken into account. The result is a number for each fact: how many points it pushed this particular prediction up or down, starting from the average case.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
sv = explainer(Xte)

# For yes/no models SHAP returns two sides; we keep the "did reoffend" side
if sv.values.ndim == 3:
    sv = sv[:, :, 1]

print("Across everyone, here is how much each fact mattered on average:")
shap.plots.bar(sv, show=True)

print("And here is the same thing spread out person by person:")
shap.plots.beeswarm(sv, show=True)

### What just happened

The first chart ranks the facts by how much they move the score, on average, across all the people in the test set. The second chart is the interesting one. Each dot is one person. Dots to the right mean "this fact pushed that person's score up, toward high risk". Colour shows whether the underlying value was high or low.

Look at where **prior convictions** and **age** land. Look at where the **race** facts land. A fact does not have to sit at the top of the chart to be doing quiet, systematic work. This is the moment to compare what you see against the answer you held in mind a few cells ago about which facts *should* count.

### Play: explain one person at a time

Averages hide individuals. Drag the slider to pick a single person from the test set. You will see the model's score for them, and a breakdown of which facts pushed their score up (to the right) and which pulled it down (to the left).

Try to find a person the model flags as high risk mostly because of their record, and another it flags mostly because of facts they did not choose.

In [ ]:
def show_person(i):
    p = model.predict_proba(Xte.iloc[[i]])[0, 1]
    verdict = "HIGH risk" if p >= 0.5 else "lower risk"
    print(f"Person #{i}   model score = {p:.2f}   ->   {verdict}")
    print("Each bar below is one fact. Right pushes toward high risk, left pulls away.")
    shap.plots.waterfall(sv[i], show=True)

interact(show_person,
         i=widgets.IntSlider(min=0, max=len(Xte) - 1, step=1, value=0,
                             description="Person #", continuous_update=False))

### What just happened

That breakdown is what people mean when they say a model is "explainable". For any single decision, SHAP hands you a tidy story: this person scored high because of these facts, despite those. It feels like a reason.

Hold on to a small suspicion, though. SHAP is telling you how *this model's* output responds to *these inputs*. It is faithful to the model. Whether the model itself is tracking anything real about a human being is a separate question, and not one SHAP can answer.

### Think first

The model outputs a number between 0 and 1. Somebody still has to decide where to draw the line for "high risk". Set the line low and you flag almost everyone. Set it high and you flag almost no one.

- If you raise the bar, what happens to the people who get wrongly flagged?
- What happens, at the same time, to the people who are wrongly cleared?
- Is there a setting that is fair to everyone at once? Is there a neutral place to stand?

<details>
<summary><b>Hint:</b> if every cutoff wrongs someone, is any of them neutral?</summary>

Raise the bar and you flag fewer people, so fewer are wrongly flagged, but more people who do go on to reoffend slip through as wrongly cleared. Lower it and the trade runs the other way. You are moving the harm around, not removing it. And no, there is no neutral place to stand: every cutoff encodes a choice about which mistake you would rather make, which is a value judgement and not something the data can settle for you. Watch the per-group numbers as you slide, because a cutoff that looks balanced overall can still flag one group far more than another.

</details>

In [ ]:
proba_te = model.predict_proba(Xte)[:, 1]

def explore_threshold(cutoff=0.5):
    flagged = proba_te >= cutoff
    fp = int(np.sum(flagged & (yte == 0)))   # flagged but did not reoffend
    fn = int(np.sum((~flagged) & (yte == 1)))  # cleared but did reoffend

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(proba_te[yte == 0], bins=20, alpha=0.6, label="did not reoffend")
    ax.hist(proba_te[yte == 1], bins=20, alpha=0.6, label="did reoffend")
    ax.axvline(cutoff, color="black", linestyle="--", linewidth=2,
               label=f"your cutoff = {cutoff:.2f}")
    ax.set_xlabel("model risk score")
    ax.set_ylabel("number of people")
    ax.legend()
    plt.show()

    print(f"Flagged as high risk : {int(flagged.sum())} of {len(yte)} people")
    print(f"Wrongly flagged      : {fp}  (did NOT reoffend, but flagged)")
    print(f"Wrongly cleared      : {fn}  (DID reoffend, but not flagged)")
    print("\nShare of each group flagged as high risk at this cutoff:")
    for g in ["African-American", "Caucasian", "Hispanic"]:
        m = race_te == g
        if m.sum() > 0:
            print(f"   {g:18s} {flagged[m].mean():.0%}")

interact(explore_threshold,
         cutoff=widgets.FloatSlider(min=0.10, max=0.90, step=0.05, value=0.50,
                                    description="cutoff", continuous_update=False))

### What just happened

Two things, both worth sitting with.

First, the trade is unavoidable. Every place you can put the line trades wrongly-flagged people against wrongly-cleared people. You cannot drive both to zero. Choosing a cutoff is not a technical step you can offload to the model. It is a value judgement about which mistake you are more willing to make, and that judgement belongs to a person who can be held responsible for it.

Second, look at the per-group numbers as you slide. At most cutoffs the share flagged is not the same across groups. That gap is the heart of the ProPublica argument. Notice that no explanation tool produced this problem and none of them will dissolve it. SHAP can tell you *that* race influences a score. It cannot tell you whether it *should*. That part is yours.

## Part 2: Which words convinced it?

Now we move from numbers to language. We will train a model to tell apart posts from two old internet forums, one for atheists and one for Christians, using nothing but the words in each post. Then we will ask a tool called **LIME** to point at the words that drove each verdict.

For philosophers of religion this is a slightly mischievous setup. The model has no theology, no arguments, no sense of what is at stake. It has only word frequencies. Watching what it leans on is a good way to see the gap between *recognising a pattern* and *understanding a claim*.

### Think first

A post is going to be sorted into "atheism" or "christianity" based on its words alone.

- Which words do you expect to drag a post toward "christianity"?
- Which toward "atheism"?
- Do you think the deciding words will be the deep ones (faith, evidence, salvation) or the boring ones nobody notices?

<details>
<summary><b>Hint:</b> would you have offered those words as a reason yourself?</summary>

Expect the boring words to win. A model like this keys on whatever separates the two forums statistically, which is often function words, usernames, and quirks of formatting rather than the deep vocabulary of faith, evidence, and salvation you might predict. That is the lesson, not a failure. Spotting a pattern in word frequencies is not the same as understanding a claim. When you run LIME in a moment, check whether the words it blames are ones you would ever have offered as a reason yourself.

</details>

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

cats = ["alt.atheism", "soc.religion.christian"]
train = fetch_20newsgroups(subset="train", categories=cats,
                           remove=("headers", "footers", "quotes"))
test = fetch_20newsgroups(subset="test", categories=cats,
                          remove=("headers", "footers", "quotes"))
class_names = ["atheism", "christianity"]

text_clf = make_pipeline(
    TfidfVectorizer(min_df=3, stop_words="english"),
    LogisticRegression(max_iter=1000),
)
text_clf.fit(train.data, train.target)
print(f"This text model sorts about {text_clf.score(test.data, test.target):.0%} of held-out posts correctly.")

### LIME, in one idea

Suppose a dish tastes too salty and you want to know which ingredient is responsible. One honest method: remove ingredients one at a time and taste again. The ingredient whose removal changes the taste most is the culprit.

**LIME** does exactly this to a piece of text. It takes the post, hides some words, and watches how the model's guess shifts. Hide a word and the "christianity" score drops a lot? That word was carrying weight. Out of many such pokes LIME builds a simple, local story: for *this* post, these were the words that mattered, and in which direction. The name stands for Local Interpretable Model-agnostic Explanations, which is a mouthful for "a small honest story about one decision".

### Play: explain a post

Pick one of the example posts from the dropdown, or paste your own text into the box, then press **Run interact**. You will see the model's guess and the words LIME blames for it. Green words pushed toward one class, red toward the other.

Try writing two sentences of your own that *you* think are obviously about faith, and see whether the words LIME picks are the ones you would have pointed to.

In [ ]:
from lime.lime_text import LimeTextExplainer

explainer_txt = LimeTextExplainer(class_names=class_names)
nonempty = [t for t in test.data if len(t.strip()) > 200]
examples = {f"Example post {k + 1}": nonempty[k] for k in range(3)}

def explain_text(choice, your_own=""):
    text = your_own.strip() if your_own.strip() else examples[choice]
    print(text[:500] + ("..." if len(text) > 500 else ""))
    print("-" * 60)
    probs = text_clf.predict_proba([text])[0]
    for name, p in zip(class_names, probs):
        print(f"   guess {name:13s} {p:.0%}")
    exp = explainer_txt.explain_instance(text, text_clf.predict_proba, num_features=10)
    exp.as_pyplot_figure()
    plt.tight_layout()
    plt.show()

interact_manual(
    explain_text,
    choice=widgets.Dropdown(options=list(examples.keys()), description="example"),
    your_own=widgets.Textarea(value="", description="or type:",
                              placeholder="paste your own sentence here",
                              layout=widgets.Layout(width="80%", height="80px")),
)

### What just happened, and the catch

LIME hands you a list of words and calls them the reasons. Often they are reasonable. Often they are not: a stray name, a quirk of punctuation, a word that just happens to be common in one forum. The model will sort a post correctly and still "explain" itself by pointing at something silly. That is not a bug in LIME. It is LIME being honest about a model that was never reasoning in the first place.

Here is the philosophical knot. In 1960s split-brain experiments, Michael Gazzaniga and his colleagues found that people would confidently explain choices their conscious mind had not actually made, inventing a reason on the spot that felt entirely sincere. Philosophers since Anscombe have pressed on the difference between the cause of an action and the reason offered for it. LIME gives you the model's offered reason. Whether that is the same thing as the cause of the model's output is exactly the kind of question you are trained to ask. A plausible explanation and a true one are not the same, and the gap between them does not announce itself.

## Part 3: Where is it looking?

Our last model sees images instead of reading them. We will use a standard image recogniser that was trained on a thousand everyday categories, hand it a photo of a cat, and then ask: when it decided "cat", which parts of the picture was it actually relying on?

The tools here are called **saliency maps** and **Grad-CAM**. They produce a heat map over the image, brighter where the pixels mattered more to the decision. For a philosopher this raises a old question in new clothes. When we say the model is "looking" at the ears, are we describing perception, or just borrowing a word that does not really fit?

### Think first

The model is about to recognise a cat.

- Which parts of the photo do you expect it to lean on? The eyes? The ears? The whole silhouette? The texture of the fur?
- Could it get the right answer for the wrong reason, by reading something in the background instead of the animal?

<details>
<summary><b>Hint:</b> where it looked is not the same as what it grasped.</summary>

You will probably say the face, the ears, the eyes, and Grad-CAM often does land there. But often is not always, and landing on the animal is not the same as grasping it. And yes, a model can be right for the wrong reason: the classic case is a classifier that picked "wolf" by noticing snow in the background instead of the animal. A heat map is good at catching that kind of cheat and useless at telling you the model knows what a cat is. Keep the difference between <em>where</em> it looked and <em>what</em> it understood in view.

</details>

In [ ]:
import torch
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from skimage import data as skdata
from PIL import Image

weights = ResNet18_Weights.IMAGENET1K_V1
net = resnet18(weights=weights).eval()
categories = weights.meta["categories"]

img_np = skdata.chelsea()            # a built-in photo of a cat
pil_img = Image.fromarray(img_np)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
inp = preprocess(pil_img).unsqueeze(0)

with torch.no_grad():
    probs = torch.softmax(net(inp), dim=1)[0]
top5 = probs.topk(5)

plt.imshow(img_np); plt.axis("off"); plt.title("the photo we will explain"); plt.show()
print("The model's top guesses:")
for p, c in zip(top5.values, top5.indices):
    print(f"   {categories[int(c)]:25s} {p.item():.0%}")
pred_idx = int(top5.indices[0])

### What just happened

The model ranked its guesses for what the photo contains. Now we open it up. The next cell lets you choose among three ways of asking "where did it look", and to dial how strongly the heat map is painted over the photo.

- **Saliency (gradients)** marks the individual pixels that the decision was most sensitive to. It tends to look speckly.
- **Integrated Gradients** is a more careful cousin that compares the photo against a blank baseline. Usually cleaner.
- **Grad-CAM** works at a coarser level and highlights whole regions the deeper layers responded to. This is the one that usually lands on the animal.

### Play: pick a method and see where it looked

Choose a method from the dropdown, set the overlay strength, and press **Run interact**.

In [ ]:
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, LayerAttribution

base_img = np.array(pil_img.resize((224, 224)))

def show_attention(method="Grad-CAM (regions)", overlay=0.5):
    x = inp.clone().requires_grad_(True)
    if method == "Grad-CAM (regions)":
        gc = LayerGradCam(net, net.layer4)
        attr = gc.attribute(x, target=pred_idx)
        attr = LayerAttribution.interpolate(attr, (224, 224))
        heat = attr[0].mean(0).detach().numpy()
    elif method == "Integrated Gradients":
        attr = IntegratedGradients(net).attribute(x, target=pred_idx, n_steps=30)
        heat = attr[0].abs().mean(0).detach().numpy()
    else:
        attr = Saliency(net).attribute(x, target=pred_idx)
        heat = attr[0].abs().mean(0).detach().numpy()

    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-9)
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(base_img); ax[0].axis("off"); ax[0].set_title("photo")
    ax[1].imshow(base_img); ax[1].imshow(heat, cmap="jet", alpha=overlay)
    ax[1].axis("off"); ax[1].set_title(f"{method}\nbright = mattered more")
    plt.show()

interact_manual(
    show_attention,
    method=widgets.Dropdown(options=["Grad-CAM (regions)", "Integrated Gradients",
                                     "Saliency (gradients)"], description="method"),
    overlay=widgets.FloatSlider(min=0.1, max=0.9, step=0.1, value=0.5, description="overlay"),
)

### What just happened

Switch between the three methods and you will notice they do not fully agree. They are answering slightly different questions, and "where the model looked" turns out not to have one clean answer. That alone is worth pausing on, because it means the heat map is a description we chose, not a fact we read off.

Now the deeper point. Even when Grad-CAM lands neatly on the cat's face, it tells you *where* in the image the decision was sensitive, not *what* the model grasped. There is a famous cautionary tale, sometimes called the Clever Hans problem after a horse that "did arithmetic" by reading its trainer's body language, and echoed in real models that classified huskies as wolves because they had learned to spot snow in the background. A saliency map is very good at catching that kind of cheat. It is no good at all at telling you the model understands what a cat is. Calling this "attention" or "looking" imports a whole philosophy of mind that the pixels have not earned.

## Part 4: So, should we believe the explanation?

You have now seen three explanation tools, one per kind of data:

| Tool | Works on | The question it answers | How it works |
|------|----------|-------------------------|--------------|
| SHAP | tables of facts | how much did each fact move this score | fair-share accounting from game theory |
| LIME | text | which words drove this verdict | hide parts of the input and watch the guess shift |
| Saliency / Grad-CAM | images | where in the picture did the decision live | follow the gradient back to the pixels or regions |

They split into two families. SHAP and LIME treat the model as a black box and probe it from outside. Saliency and Grad-CAM reach inside and read the gradients. Different machinery, same promise: here is why.

### The faithfulness problem

Every tool in this lab can produce an explanation that is plausible and still wrong. "Plausible" means it reads well to a human. "Wrong" means it does not match what the model actually did. The two come apart more often than the polished plots suggest, and nothing in the output flags which kind you are looking at.

This should feel familiar. It is the reasons-versus-causes problem from the start of the lab, now with stakes. A defendant told "the algorithm flagged you because of your priors" deserves to know whether that is the real cause of the score or a comfortable story laid over it. An explanation that cannot be checked is not yet a justification. It is a claim about a justification.

So the tools are necessary and not sufficient. They give you something to interrogate, which is far better than a bare number. They do not hand you permission to trust.

### Final reflection

Write a few sentences on each, and try to answer before you peek. There are no marks here and no single right answer; each reveal is just one defensible line to argue with, not the answer.

1. **Reasons or rationalisations.** SHAP, LIME, and Grad-CAM all produce something that looks like a reason. Pick one and argue whether it gives a genuine reason for the model's decision or only a plausible story about it. What evidence would settle the question?

<details>
<summary><b>Hint:</b> does the explanation predict new behaviour, or only fit what already happened?</summary>

Take LIME. Honestly, on its own it gives you a plausible story, not a proven reason. It reports which words, when hidden, move the model's output, and that is a real fact about the model. But <em>this word swung the score</em> is a claim about sensitivity, not about anything the model understood or intended, so calling it the reason quietly claims more than LIME earned. What would settle it? A faithfulness test. Perturb the input in the ways the explanation says should matter and check the output really moves, then perturb in ways it says should not matter and check it stays put. If the explanation keeps predicting the model's behaviour under fresh pokes, it has earned the word reason. If it only ever sounds right after the fact, it is a rationalisation. The split-brain patients from Part 2 are the warning: a sincere, fluent explanation can be completely cut off from the real cause.

</details>

2. **The cutoff is yours.** In Part 1 you moved the risk threshold by hand and watched the trade between wrongly-flagged and wrongly-cleared people shift, unevenly across groups. Who should choose that cutoff in a real courtroom, and what would make their choice legitimate?

<details>
<summary><b>Hint:</b> who can actually be held responsible for the choice?</summary>

There is no value-free cutoff, so the real question is who may make a value judgement that lands on other people, and how they answer for it. A defensible line: the choice belongs to an accountable public body, a legislature or a court acting under published rules, not to the vendor who built the tool and not to a single judge improvising in the moment. Legitimacy comes from three things at once. The choice is made in the open, with the trade between wrongly-flagged and wrongly-cleared stated plainly. The people who bear the cost have some voice in setting it. And whoever sets it can be questioned and removed. That rules out the quiet default: shipping the model at 0.5 is not a justification, it is an abdication dressed up as a technical setting.

</details>

3. **Looking without seeing.** Grad-CAM showed where the image model was sensitive. Does "the model looked at the ears" describe perception, or is it a metaphor we should retire? Defend your line.

<details>
<summary><b>Hint:</b> what does looking carry that a gradient does not?</summary>

I would keep the phrase but demote it to shorthand. Saying the model looked at the ears is a metaphor, and a loaded one, because looking in the full sense drags in attention, a point of view, someone who sees. Grad-CAM delivers none of that. It shows which regions the output was numerically sensitive to, which is a fact about gradients, not about experience. The honest paraphrase is that the decision was most sensitive to this region, which is duller and truer. So retire the metaphor wherever precision matters, in a courtroom, a paper, an audit, and keep it only as casual shorthand among people who already know it is shorthand. The real danger is not the word. It is forgetting the word was ever a metaphor.

</details>

4. **When is an explanation good enough?** Philosophers of science have long asked what makes an explanation a good one. Borrow one standard you find convincing, from Hempel, from Wesley Salmon's causal account, from anywhere, and judge these three tools against it. Do any of them clear the bar?

<details>
<summary><b>Hint:</b> pick your standard first, then judge the tools against it.</summary>

Take Salmon's causal-mechanical standard: a good explanation traces the actual causal process behind the outcome, not just a pattern that happens to fit. By that bar all three tools struggle. SHAP and LIME describe how the output covaries with the inputs, which is closer to Hempel's covering-law idea, a statistical regularity, than to real mechanism. Saliency and Grad-CAM get a little nearer the machinery, since gradients are part of how the network actually computes, but this region had a high gradient is still not a causal story about why the concept was recognised. Switch to Hempel and the tools look better, because a stable statistical dependence is roughly what he asked for. That is the honest verdict: these tools clear a low bar for explanation and fail a high one, and half the work is being clear about which bar you are holding them to.

</details>

5. **The right to an explanation.** Some laws now grant people a "right to an explanation" when an automated system decides about them. After this lab, is that right worth much? What would have to be true of the explanation for it to mean something?

<details>
<summary><b>Hint:</b> what is a right to a story worth if the story may be false?</summary>

As usually written, the right is worth less than it sounds, because it rarely says what the explanation has to be, and this lab showed that a plausible explanation can be flatly unfaithful to the model. A right to be handed a story is worth little if the story is allowed to be false. To mean something, the explanation would have to be faithful to what the model did, testable against its behaviour rather than merely nice to read, specific to your case rather than a stock paragraph, and actionable, telling you what would have had to be different for the decision to flip. Pair it with a way to contest the decision in front of someone who can overturn it. Short of that, the right hands you the feeling of an account without the substance of one, which is arguably worse than nothing, because it looks like accountability and is not.

</details>

**Your answers:**

1.

2.

3.

4.

5.

## Wrap-up

What you did today:

- Made a recidivism model **explain individual verdicts** with SHAP, and watched a risk threshold turn into a value judgement with uneven effects across groups.
- Made a text model **name the words** behind its calls with LIME, and met the gap between a plausible reason and a true one.
- Made an image model **show where it looked** with saliency maps and Grad-CAM, and saw why "looking" is a loaded word.
- Hit the **faithfulness problem** head on: an explanation can read well and still misdescribe the thing it explains.

The engineering question was "how do we explain a model". The question this lab leaves you with is older and harder: what do we owe a person when a machine decides something about them, and when does an explanation become a justification.

### Where this sits in the course
- It picks up the fairness thread from the **COMPAS** labs and gives it a new tool.
- It is the outward-facing companion to the interpretability labs: the **Logit Lens** and **sparse autoencoders** open the model up from the inside, while LIME, SHAP, and saliency question it from the outside.

---

**Author:** [Aniket Ghosh](https://www.linkedin.com/in/aniketghosh-/)